# T31-机构行为监测 - 测试验证模块

## 1. 模块概述

测试验证模块提供单元测试和集成测试，确保各模块功能正确性。

包含：
- 数据清洗测试
- 算法计算测试
- 数据格式测试
- 端到端流程测试

## 2. 测试框架设置

In [ ]:
import pandas as pd
import numpy as np
import datetime
from typing import Dict, List, Any
import unittest
from io import StringIO
import sys

# 测试结果收集器
test_results = []

def run_test(test_name: str, test_func):
    """运行单个测试并记录结果"""
    try:
        test_func()
        test_results.append({'name': test_name, 'status': 'PASSED', 'error': None})
        print(f"[PASS] {test_name}")
        return True
    except AssertionError as e:
        test_results.append({'name': test_name, 'status': 'FAILED', 'error': str(e)})
        print(f"[FAIL] {test_name}: {e}")
        return False
    except Exception as e:
        test_results.append({'name': test_name, 'status': 'ERROR', 'error': str(e)})
        print(f"[ERROR] {test_name}: {e}")
        return False

print("测试框架已初始化")

## 3. 模拟测试数据

In [ ]:
def create_mock_institution_stats() -> pd.DataFrame:
    """创建模拟机构统计数据"""
    np.random.seed(42)
    
    institutions = ['大型商业银行/政策性银行', '股份制商业银行', '基金公司及产品', '证券公司', '保险公司']
    tenors = ['1Y', '3Y', '5Y', '10Y']
    bond_types = ['国债', '金融债', '中期票据', '企业债']
    dates = pd.date_range('2025-01-01', '2025-01-31', freq='D')
    
    data = []
    for date in dates:
        for inst in institutions:
            for tenor in tenors:
                for bond_type in bond_types:
                    buy_vol = np.random.uniform(10, 100)
                    sell_vol = np.random.uniform(10, 100)
                    data.append({
                        '交易日期': date,
                        '机构类型': inst,
                        '期限': tenor,
                        '债券类型': bond_type,
                        '买入交易量（亿元）': buy_vol,
                        '卖出交易量（亿元）': sell_vol,
                        '净买入交易量（亿元）': buy_vol - sell_vol
                    })
    
    return pd.DataFrame(data)

def create_mock_dealtinfo() -> pd.DataFrame:
    """创建模拟交易明细数据"""
    np.random.seed(42)
    
    n = 1000
    data = {
        'DT': np.random.choice(pd.date_range('2025-01-01', '2025-01-31'), n),
        'TRADE_CODE': [f'BOND{i:04d}' for i in range(n)],
        'SEC_NAME': [f'债券{i:03d}' for i in np.random.randint(1, 101, n)],
        'transaction_amount': np.random.uniform(100, 10000, n),
        'remaining_term': np.random.choice(['1Y+', '3Y+', '5Y+', '10Y+', '180D'], n),
        'weighted_YTM': np.random.uniform(0.02, 0.05, n)
    }
    
    return pd.DataFrame(data)

# 创建测试数据
mock_stats = create_mock_institution_stats()
mock_dealtinfo = create_mock_dealtinfo()

print(f"模拟机构统计数据: {mock_stats.shape}")
print(f"模拟交易明细数据: {mock_dealtinfo.shape}")

## 4. 数据清洗测试

In [ ]:
print("=" * 60)
print("数据清洗测试")
print("=" * 60)

# 测试1: 特殊值清洗
def test_clean_special_values():
    df = pd.DataFrame({
        'value': ['10', '--', '20', '--', '30']
    })
    
    # 模拟清洗
    df['value'] = df['value'].replace('--', 0)
    df['value'] = pd.to_numeric(df['value'])
    
    assert df['value'].tolist() == [10, 0, 20, 0, 30], "特殊值清洗失败"
    assert df['value'].dtype in [np.float64, np.int64], "类型转换失败"

run_test("test_clean_special_values", test_clean_special_values)

# 测试2: 日期清洗
def test_clean_date_column():
    df = pd.DataFrame({
        '交易日期': ['2025-01-15', '2025-01-16', '2025-01-17']
    })
    
    df['交易日期'] = pd.to_datetime(df['交易日期'])
    
    assert pd.api.types.is_datetime64_any_dtype(df['交易日期']), "日期类型转换失败"
    assert len(df) == 3, "数据行数不正确"

run_test("test_clean_date_column", test_clean_date_column)

# 测试3: 空值处理
def test_handle_null_values():
    df = pd.DataFrame({
        'value': [1, np.nan, 3, np.nan, 5]
    })
    
    df_clean = df.dropna()
    
    assert len(df_clean) == 3, "空值删除不正确"
    assert df_clean['value'].tolist() == [1, 3, 5], "删除后数据不正确"

run_test("test_handle_null_values", test_handle_null_values)

## 5. 期限解析测试

In [ ]:
print("\n" + "=" * 60)
print("期限解析测试")
print("=" * 60)

import re

def parse_tenor(tenor_str):
    """解析期限字符串"""
    if tenor_str is None or pd.isna(tenor_str):
        return None
    
    tenor_str = str(tenor_str).strip()
    
    if 'Y' in tenor_str:
        match = re.match(r'(\d+\.?\d*)Y', tenor_str)
        if match:
            return float(match.group(1))
    
    if 'D' in tenor_str:
        match = re.match(r'(\d+\.?\d*)D', tenor_str)
        if match:
            return float(match.group(1)) / 365
    
    return None

# 测试1: 年限格式
def test_parse_year_tenor():
    assert parse_tenor('1Y+') == 1.0, "1Y+解析失败"
    assert parse_tenor('5Y+') == 5.0, "5Y+解析失败"
    assert parse_tenor('10Y+') == 10.0, "10Y+解析失败"
    assert parse_tenor('3Y') == 3.0, "3Y解析失败"

run_test("test_parse_year_tenor", test_parse_year_tenor)

# 测试2: 天数格式
def test_parse_day_tenor():
    assert abs(parse_tenor('365D') - 1.0) < 0.01, "365D解析失败"
    assert abs(parse_tenor('180D') - 0.493) < 0.01, "180D解析失败"

run_test("test_parse_day_tenor", test_parse_day_tenor)

# 测试3: 边界情况
def test_parse_edge_cases():
    assert parse_tenor(None) is None, "None解析失败"
    assert parse_tenor('') is None, "空字符串解析失败"
    assert parse_tenor('invalid') is None, "无效格式解析失败"

run_test("test_parse_edge_cases", test_parse_edge_cases)

## 6. 聚合计算测试

In [ ]:
print("\n" + "=" * 60)
print("聚合计算测试")
print("=" * 60)

# 测试1: 机构聚合
def test_aggregate_by_institution():
    df = mock_stats.copy()
    
    # 按机构聚合净买入量
    result = df.groupby('机构类型')['净买入交易量（亿元）'].sum().reset_index()
    
    assert len(result) == 5, "机构聚合结果行数不正确"
    assert '机构类型' in result.columns, "缺少机构类型列"
    assert '净买入交易量（亿元）' in result.columns, "缺少值列"

run_test("test_aggregate_by_institution", test_aggregate_by_institution)

# 测试2: 市场份额计算
def test_calculate_market_share():
    df = mock_stats.copy()
    
    # 计算总交易量
    df['总交易量'] = df['买入交易量（亿元）'] + df['卖出交易量（亿元）']
    
    # 按日期计算市场总量
    daily_total = df.groupby('交易日期')['总交易量'].sum().reset_index()
    daily_total.rename(columns={'总交易量': '市场总量'}, inplace=True)
    
    assert '市场总量' in daily_total.columns, "市场总量计算失败"
    assert len(daily_total) == 31, "日期数量不正确"
    assert (daily_total['市场总量'] > 0).all(), "市场总量应大于0"

run_test("test_calculate_market_share", test_calculate_market_share)

# 测试3: 百分比计算
def test_calculate_percentage():
    df = pd.DataFrame({
        'name': ['A', 'B', 'C', 'D'],
        'value': [25, 25, 25, 25]
    })
    
    total = df['value'].sum()
    df['percent'] = (df['value'] / total) * 100
    
    assert df['percent'].sum() == 100.0, "百分比总和应为100"
    assert (df['percent'] == 25.0).all(), "每个项目应占25%"

run_test("test_calculate_percentage", test_calculate_percentage)

## 7. 算法计算测试

In [ ]:
print("\n" + "=" * 60)
print("算法计算测试")
print("=" * 60)

# 测试1: 加权平均计算
def test_weighted_average():
    df = pd.DataFrame({
        'value': [1, 2, 3, 4],
        'weight': [1, 2, 3, 4]
    })
    
    weighted_sum = (df['value'] * df['weight']).sum()
    total_weight = df['weight'].sum()
    weighted_avg = weighted_sum / total_weight
    
    expected = (1*1 + 2*2 + 3*3 + 4*4) / 10  # = 3.0
    assert abs(weighted_avg - expected) < 0.001, f"加权平均计算错误: {weighted_avg} != {expected}"

run_test("test_weighted_average", test_weighted_average)

# 测试2: 利差计算
def test_spread_calculation():
    yield_credit = 0.045  # 4.5%
    yield_gov = 0.025     # 2.5%
    
    spread_bp = (yield_credit - yield_gov) * 100  # BP
    
    expected = 200  # 200 BP
    assert abs(spread_bp - expected) < 0.1, f"利差计算错误: {spread_bp} != {expected}"

run_test("test_spread_calculation", test_spread_calculation)

# 测试3: 净流入计算
def test_net_inflow_calculation():
    buy_vol = 100
    sell_vol = 80
    
    net_inflow = buy_vol - sell_vol
    
    assert net_inflow == 20, f"净流入计算错误: {net_inflow} != 20"

run_test("test_net_inflow_calculation", test_net_inflow_calculation)

## 8. 数据格式测试

In [ ]:
print("\n" + "=" * 60)
print("数据格式测试")
print("=" * 60)

# 测试1: 柱状图数据格式
def test_bar_chart_format():
    df = pd.DataFrame({
        'name': ['A', 'B', 'C'],
        'value': [10, 20, 30]
    })
    
    chart_data = {
        'categories': df['name'].tolist(),
        'values': df['value'].tolist()
    }
    
    assert chart_data['categories'] == ['A', 'B', 'C'], "categories格式错误"
    assert chart_data['values'] == [10, 20, 30], "values格式错误"

run_test("test_bar_chart_format", test_bar_chart_format)

# 测试2: 饼图数据格式
def test_pie_chart_format():
    df = pd.DataFrame({
        'name': ['A', 'B', 'C'],
        'value': [10, 20, 30],
        'percent': [16.67, 33.33, 50.0]
    })
    
    pie_data = []
    for _, row in df.iterrows():
        pie_data.append({
            'name': row['name'],
            'value': row['value'],
            'percent': row['percent']
        })
    
    assert len(pie_data) == 3, "饼图数据项数量错误"
    assert all('name' in item and 'value' in item for item in pie_data), "饼图数据项缺少必要字段"

run_test("test_pie_chart_format", test_pie_chart_format)

# 测试3: 热力图数据格式
def test_heatmap_format():
    matrix = pd.DataFrame({
        'A': [1, 2, 3],
        'B': [4, 5, 6]
    }, index=['X', 'Y', 'Z'])
    
    heatmap_data = []
    for y, row_name in enumerate(matrix.index):
        for x, col_name in enumerate(matrix.columns):
            heatmap_data.append({
                'x': x,
                'y': y,
                'value': matrix.loc[row_name, col_name]
            })
    
    assert len(heatmap_data) == 6, "热力图数据点数量错误"
    assert all('x' in item and 'y' in item and 'value' in item for item in heatmap_data), "热力图数据项缺少必要字段"

run_test("test_heatmap_format", test_heatmap_format)

## 9. 端到端测试

In [ ]:
print("\n" + "=" * 60)
print("端到端测试")
print("=" * 60)

# 测试: 完整处理流程
def test_full_pipeline():
    # 1. 数据加载
    df = mock_stats.copy()
    assert not df.empty, "数据加载失败"
    
    # 2. 数据清洗
    volume_cols = ['买入交易量（亿元）', '卖出交易量（亿元）', '净买入交易量（亿元）']
    for col in volume_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    assert df[volume_cols].notna().all().all(), "数据清洗失败"
    
    # 3. 聚合计算
    result = df.groupby('机构类型')['净买入交易量（亿元）'].sum().reset_index()
    result.rename(columns={'机构类型': 'name', '净买入交易量（亿元）': 'value'}, inplace=True)
    assert 'name' in result.columns and 'value' in result.columns, "聚合结果列名错误"
    
    # 4. 格式转换
    chart_data = {
        'categories': result['name'].tolist(),
        'values': result['value'].round(2).tolist()
    }
    assert 'categories' in chart_data and 'values' in chart_data, "图表数据格式错误"
    
    print("  完整流程测试成功")

run_test("test_full_pipeline", test_full_pipeline)

## 10. 测试结果汇总

In [ ]:
print("\n" + "=" * 60)
print("测试结果汇总")
print("=" * 60)

# 统计结果
total = len(test_results)
passed = sum(1 for r in test_results if r['status'] == 'PASSED')
failed = sum(1 for r in test_results if r['status'] == 'FAILED')
errors = sum(1 for r in test_results if r['status'] == 'ERROR')

print(f"\n总测试数: {total}")
print(f"通过: {passed}")
print(f"失败: {failed}")
print(f"错误: {errors}")
print(f"通过率: {passed/total*100:.1f}%" if total > 0 else "N/A")

# 详细结果
print("\n详细测试结果:")
for r in test_results:
    status_icon = "OK" if r['status'] == 'PASSED' else "X"
    print(f"  [{status_icon}] {r['name']}: {r['status']}")
    if r['error']:
        print(f"      Error: {r['error']}")

# 测试报告
test_report = {
    'summary': {
        'total': total,
        'passed': passed,
        'failed': failed,
        'errors': errors,
        'pass_rate': round(passed/total*100, 1) if total > 0 else 0
    },
    'details': test_results
}

print("\n" + "=" * 60)
print(f"测试验证完成 - 通过率: {test_report['summary']['pass_rate']}%")
print("=" * 60)

In [ ]:
# 导出测试报告
import json

print("测试报告JSON:")
print(json.dumps(test_report, indent=2, ensure_ascii=False))